# Build `Euclid-Q1-stamps64` from Google Drive (Colab)

Builds the postage-stamp dataset from the DET / BKG / PSF / catalogue files already stored in
`MyDrive/Q1_VIS_CALIBRATED_DB`, without re-downloading the frames, and pushes it to the Hugging Face Hub.

1. mount the Drive, clone the repo, read the Hugging Face token
2. keep the observations whose DET + BKG + catalogue are on the Drive
   (a missing catalogue is downloaded when `FETCH_MISSING_CATALOGUES = True`)
3. slice the per-quadrant FITS into `quadrant-data/` on the Drive (already sliced quadrants are skipped)
4. build + push in batches of `BATCH_SIZE` observations: the first batch with `--push`, the next ones
   with `--merge`, all with `--zero-flagged-pixels --drop-duplicates --drop-truncated` and the
   `psf_residual` column (reference PSF `src/euclid_vis_isotropic_min_psf.fits`)

**After a disconnection, just run all the cells again**: incomplete quadrant files are deleted and
re-sliced, and the batches already pushed (listed in `PROGRESS_FILE` on the Drive) are skipped.
To rebuild the dataset from scratch, delete `PROGRESS_FILE`.

The Hugging Face token (write access) is read from the Colab secret `HF_TOKEN`, or asked for.

In [ ]:
# ---- Parameters -----------------------------------------------------------------------------
OBS_IDS = [
    2696, 2684, 2683, 2682, 2697, 2695, 2685, 2681, 2686, 2698, 2701, 2694, 2687, 2688, 2693, 2699,
    2700, 2702, 2692, 2691, 2703, 2689, 2715, 2714, 2716, 2704, 2705, 2713, 2690, 2723, 2717, 2712,
    2711, 2718, 2706, 2722, 2719, 2710, 2707, 2721, 2720, 2709, 2708, 65714, 65744, 3033, 3032, 3023,
    3034, 3022, 3031, 3024, 3035, 3021, 3030, 3015, 3025, 3036, 3020, 3029, 3016, 3026, 3019, 3037,
    3017, 3028, 3027, 3018, 3623, 3624, 3611, 3625, 3610, 3622, 3626, 3599, 3612, 3630, 3627, 3609,
]

REPO_ID = "VincentB03/Euclid-Q1-stamps64"
PUBLIC = False                    # False = private dataset
BATCH_SIZE = 20                   # observations per build + push
FETCH_MISSING_CATALOGUES = True   # download the catalogue of observations whose DET + BKG are on the Drive

DRIVE_DATA_DIR = "/content/drive/MyDrive/Q1_VIS_CALIBRATED_DB"
PROGRESS_FILE = f"{DRIVE_DATA_DIR}/progress_{REPO_ID.split('/')[-1]}.json"

GIT_URL = "https://github.com/VincentB03/Euclid-Q1-postage-stamps.git"
REPO_DIR = "/content/Euclid-Q1-postage-stamps"

## 1. Drive, repository, Hugging Face token

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.isdir(DRIVE_DATA_DIR), f"Drive folder not found: {DRIVE_DATA_DIR!r}"
print("Data directory:", DRIVE_DATA_DIR)

In [ ]:
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {GIT_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import getpass

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write access): ")

from huggingface_hub import whoami
print("Hugging Face user:", whoami(token=os.environ["HF_TOKEN"])["name"])

In [ ]:
import collections
import json
import shutil
import subprocess
import sys

QUADRANT_DIR = f"{DRIVE_DATA_DIR}/quadrant-data"
N_QUADRANTS = 144
HEADER_SLACK = 10 * 2880   # bytes: tolerated header-size difference between quadrant files

ENV = {
    **os.environ,
    "EUCLID_DATA_DIR": DRIVE_DATA_DIR,
    "PYTHONUNBUFFERED": "1",
    "TQDM_DISABLE": "1",
    "HF_DATASETS_DISABLE_PROGRESS_BARS": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
}


def pad(obs_id):
    return str(obs_id).zfill(6)


def run_main(*args):
    """Run src/main.py on the Drive data, stream its output, raise if it fails."""
    cmd = [sys.executable, "src/main.py", *map(str, args)]
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"src/main.py failed (exit code {proc.returncode})")


def check_quadrants(obs_ids, delete_incomplete=False, require_complete=False):
    """Find quadrant files smaller than the usual size of their kind (left incomplete by a
    disconnection) and, with ``require_complete``, observations missing DET/BKG quadrants.

    Returns True when no incomplete file (and, if required, no missing quadrant) was found.
    """
    if not os.path.isdir(QUADRANT_DIR):
        return not require_complete
    entries = [e for e in os.scandir(QUADRANT_DIR) if e.name.endswith(".fits")]
    kinds = {"DET": lambda n: "-DET-" in n, "BKG": lambda n: "-BKG-" in n,
             "PSF": lambda n: n.startswith("PSF_")}
    ok = True

    for kind, is_kind in kinds.items():
        group = [(e.name, e.stat().st_size) for e in entries if is_kind(e.name)]
        if not group:
            continue
        usual = collections.Counter(size for _, size in group).most_common(1)[0][0]
        for name, size in group:
            if size < usual - HEADER_SLACK:
                ok = False
                action = "deleted" if delete_incomplete else "found"
                print(f"incomplete {kind} quadrant {action}: {name} ({size} / {usual} bytes)")
                if delete_incomplete:
                    os.remove(os.path.join(QUADRANT_DIR, name))

    if require_complete:
        for obs in obs_ids:
            tag = f"-{pad(obs)}-"
            for kind in ("DET", "BKG"):
                n = sum(1 for e in entries if kinds[kind](e.name) and tag in e.name)
                if n < N_QUADRANTS:
                    ok = False
                    print(f"obs {obs}: {n}/{N_QUADRANTS} {kind} quadrants")
    return ok

## 2. Observations available on the Drive

In [ ]:
names = set(os.listdir(DRIVE_DATA_DIR))


def on_drive(prefix):
    return any(n.startswith(prefix) for n in names)


psf_models = os.path.join(DRIVE_DATA_DIR, "psf_models")
assert on_drive("EUC_VIS_GRD-PSF-") or (
    os.path.isdir(psf_models) and any(n.startswith("EUC_VIS_GRD-PSF-") for n in os.listdir(psf_models))
), "PSF model (EUC_VIS_GRD-PSF-*.fits) not found on the Drive"

ready, need_catalogue = [], []
for obs in OBS_IDS:
    det = on_drive(f"EUC_VIS_SWL-DET-{pad(obs)}-00-1-")
    bkg = on_drive(f"EUC_VIS_SWL-BKG-{pad(obs)}-00-1-")
    cat = f"catalogue_obs_{pad(obs)}.fits" in names
    if det and bkg and cat:
        ready.append(obs)
    elif det and bkg:
        need_catalogue.append(obs)
    else:
        missing = [k for k, present in (("DET", det), ("BKG", bkg), ("catalogue", cat)) if not present]
        print(f"skip obs {obs}: missing {', '.join(missing)}")

if need_catalogue and FETCH_MISSING_CATALOGUES:
    print(f"Downloading the catalogue of {need_catalogue} ...")
    run_main("--obs-ids", *need_catalogue, "--skip-extract", "--skip-build")
    names = set(os.listdir(DRIVE_DATA_DIR))
for obs in need_catalogue:
    if f"catalogue_obs_{pad(obs)}.fits" in names:
        ready.append(obs)
    else:
        print(f"skip obs {obs}: no catalogue")

OBS = [obs for obs in OBS_IDS if obs in ready]
print(f"\n{len(OBS)} / {len(OBS_IDS)} observation(s) ready")

## 3. Slice the quadrants on the Drive

In [ ]:
check_quadrants(OBS, delete_incomplete=True)   # clean up after a previous disconnection
run_main("--obs-ids", *OBS, "--skip-acquire", "--skip-build")

if check_quadrants(OBS, require_complete=True):
    print("All quadrants are sliced.")
else:
    print("WARNING: see the messages above (an observation with missing quadrants still builds from the others).")

## 4. Build and push to the Hub, batch by batch

In [ ]:
try:
    with open(PROGRESS_FILE) as fh:
        progress = json.load(fh)
except FileNotFoundError:
    progress = {"repo_id": REPO_ID, "done": []}

todo = [obs for obs in OBS if obs not in progress["done"]]
batches = [todo[i:i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
print(f"{len(progress['done'])} observation(s) already pushed, "
      f"{len(todo)} to go in {len(batches)} batch(es)")

for i, batch in enumerate(batches, start=1):
    mode = "--merge" if progress["done"] else "--push"
    print(f"\n=== batch {i}/{len(batches)} ({mode}): {batch}")
    run_main(
        "--obs-ids", *batch,
        "--skip-acquire", "--skip-extract",
        "--zero-flagged-pixels", "--drop-duplicates", "--drop-truncated",
        "--reference-psf", "src/euclid_vis_isotropic_min_psf.fits",
        "--repo-id", REPO_ID, mode,
        *(["--public"] if PUBLIC else []),
    )
    progress["done"] += batch
    with open(PROGRESS_FILE, "w") as fh:
        json.dump(progress, fh)

    # free the Colab disk: build cache and the Hub copy downloaded by --merge
    for cache in ("~/.cache/huggingface/datasets", "~/.cache/huggingface/hub"):
        shutil.rmtree(os.path.expanduser(cache), ignore_errors=True)

print(f"\nDone: {len(progress['done'])} observation(s) in {REPO_ID}")